# MOD-02 · NB-02 — Descriptive Statistics for Clinical Variables
### Health Analytics with Python · Module 02: EDA in Healthcare
---
**Learning objectives**
- Compute prevalence, incidence, and rates with correct denominators
- Build stratified summaries by age group, sex, and payer
- Detect and handle right-skewed clinical distributions (LOS, charges)
- Apply age-standardisation (direct method) to remove age confounding
- Produce a publication-ready Table 1 with confidence intervals

**Estimated time:** 1.5 hours | **Level:** Beginner → Intermediate | **Libraries:** `pandas`, `numpy`, `scipy.stats`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind, norm
import warnings
warnings.filterwarnings('ignore')

# ── Reconstruct shared dataset ───────────────────────────────────────────────
np.random.seed(42); N = 800
ages      = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes     = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],
                              N,p=[0.40,0.22,0.28,0.07,0.03])
drg_codes = np.random.choice([470,291,392,690,871,193,247,603],N)
drg_labels= {470:'Major joint replacement',291:'Heart failure & shock',
             392:'Esophagitis/misc GI',690:'Kidney/UTI',871:'Septicemia',
             193:'Simple pneumonia',247:'Perc cardiovasc w stent',603:'Cellulitis'}
los_days  = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
charges   = (los_days*np.random.uniform(1800,4200,N)).round(2)
charges[np.random.rand(N)<0.08] = np.nan
primary_dx= np.random.choice(['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11','N39.0'],N)
admit_type= np.random.choice(['Emergency','Elective','Urgent','Newborn'],N,p=[0.52,0.30,0.16,0.02])
np.random.seed(99)
df = pd.DataFrame({
    'patient_id':   [f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,
    'drg_code':drg_codes,'drg_label':[drg_labels[d] for d in drg_codes],
    'primary_dx':primary_dx,'admit_type':admit_type,
    'los_days':los_days,'total_charge':charges,
    'diabetes':np.random.binomial(1,0.32,N),
    'hypertension':np.random.binomial(1,0.45,N),
    'ckd':np.random.binomial(1,0.15,N),
    'heart_failure':np.random.binomial(1,0.18,N),
    'readmitted_30':np.random.binomial(1,0.14,N),
})
df['age_group'] = pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])
print(f"Dataset ready: {df.shape}")


## 2. Prevalence, Incidence & Rates

In [ ]:
# ── 2.1  Comorbidity prevalence ──────────────────────────────────────────────
comorbidities = ['diabetes','hypertension','ckd','heart_failure']

def wilson_ci(k, n, alpha=0.05):
    z   = norm.ppf(1 - alpha/2)
    p   = k/n
    den = 1 + z**2/n
    mid = (p + z**2/(2*n)) / den
    hw  = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / den
    return round((mid-hw)*100,1), round((mid+hw)*100,1)

rows = []
for cond in comorbidities:
    k   = df[cond].sum()
    n   = len(df)
    lo, hi = wilson_ci(k, n)
    rows.append({'Condition':cond,'N cases':k,'Prevalence (%)':round(k/n*100,1),
                 '95% CI':f'{lo}%–{hi}%'})
prev_df = pd.DataFrame(rows).set_index('Condition')
print("Comorbidity prevalence (Wilson 95% CI):")
display(prev_df)


In [ ]:
# ── 2.2  Readmission rate per 100 discharges ────────────────────────────────
total = len(df)
read  = df['readmitted_30'].sum()
rate  = read/total*100
lo, hi = wilson_ci(read, total)
print(f"30-day readmission rate: {rate:.1f}% (95% CI: {lo}%–{hi}%)")
print(f"Absolute count: {read} of {total} discharges")


In [ ]:
# ── 2.3  Stratified readmission rate by payer ───────────────────────────────
strat = (
    df.groupby('payer')
      .apply(lambda g: pd.Series({
          'N'           : len(g),
          'Readmissions': g['readmitted_30'].sum(),
          'Rate (%)'    : round(g['readmitted_30'].mean()*100, 1),
          'Mean LOS'    : round(g['los_days'].mean(), 1),
          'Median charge': round(g['total_charge'].median(), 0),
      }))
      .sort_values('Rate (%)', ascending=False)
)
print("Readmission rate stratified by payer:")
display(strat)


## 3. Skewed Distributions — LOS and Charges

In [ ]:
from scipy.stats import skew, kurtosis

for name, col in [('LOS (days)','los_days'),('Total charge ($)','total_charge')]:
    s = df[col].dropna()
    sk = skew(s)
    print(f"{name}")
    print(f"  Mean    : {s.mean():.2f}   Median: {s.median():.2f}   Diff: {s.mean()-s.median():.2f}")
    print(f"  Std     : {s.std():.2f}")
    print(f"  Skewness: {sk:.3f}  {'→ right-skewed; use MEDIAN' if sk > 0.5 else '→ approximately symmetric'}")
    print(f"  P10–P90 : {s.quantile(.10):.1f} – {s.quantile(.90):.1f}")
    print()


In [ ]:
# Log-transform for skewed outcomes
df['log_charge'] = np.log1p(df['total_charge'])

print("Charge skewness:")
print(f"  Original  : {skew(df['total_charge'].dropna()):.3f}")
print(f"  Log+1     : {skew(df['log_charge'].dropna()):.3f}")
print()
print("Geometric mean charge (back-transformed from log):")
gm = np.exp(df['log_charge'].dropna().mean()) - 1
print(f"  ${gm:,.0f}")
print()
print("Reporting guidance:")
print("  - Report median [IQR] for LOS and charges in tables")
print("  - Use Mann-Whitney U (not t-test) for skewed outcomes")
print("  - Log-transform before Pearson correlation or linear regression")


## 4. Age-Standardisation — Direct Method

In [ ]:
# US 2000 standard population weights by age group
std_pop = pd.DataFrame({
    'age_group' : ['18-44','45-64','65-74','75+'],
    'std_weight': [0.3761, 0.3399, 0.1267, 0.0573]
})

print(f"{'Payer':12s}  {'Crude rate':>12s}  {'Age-adj rate':>12s}  {'Diff':>8s}")
print("-"*52)
for payer in ['Medicare','Private','Medicaid']:
    sub   = df[df['payer']==payer]
    crude = sub['readmitted_30'].mean()*100
    rates = (sub.groupby('age_group',observed=True)['readmitted_30']
               .mean().reset_index().rename(columns={'readmitted_30':'crude_rate'}))
    m     = rates.merge(std_pop, on='age_group')
    m['weighted'] = m['crude_rate'] * m['std_weight']
    adj   = m['weighted'].sum() * 100
    print(f"{payer:12s}  {crude:>11.1f}%  {adj:>11.1f}%  {adj-crude:>+7.1f}%")
print()
print("Note: Differences reveal age-confounding in crude comparisons.")


## 5. Publication-Ready Table 1

In [ ]:
def table1(df, stratify, continuous, categorical):
    groups = sorted(df[stratify].dropna().unique())
    records = []

    for col in continuous:
        sk = abs(skew(df[col].dropna()))
        use_median = sk > 1
        stat_label = 'median [IQR]' if use_median else 'mean ± SD'
        row = {'Variable': f'{col} ({stat_label})'}
        for g in groups:
            vals = df.loc[df[stratify]==g, col].dropna()
            if use_median:
                row[f'{stratify}={g}'] = f"{vals.median():.1f} [{vals.quantile(.25):.1f}–{vals.quantile(.75):.1f}]"
            else:
                row[f'{stratify}={g}'] = f"{vals.mean():.1f} ± {vals.std():.1f}"
        records.append(row)

    for col in categorical:
        records.append({'Variable': col})
        for cat in sorted(df[col].dropna().unique()):
            row = {'Variable': f'  {cat}'}
            for g in groups:
                sub = df[df[stratify]==g]
                n   = (sub[col]==cat).sum()
                pct = n/len(sub)*100
                row[f'{stratify}={g}'] = f"{n} ({pct:.1f}%)"
            records.append(row)

    return pd.DataFrame(records).set_index('Variable')

t1 = table1(
    df,
    stratify    = 'readmitted_30',
    continuous  = ['age','los_days','total_charge'],
    categorical = ['sex','payer','admit_type','age_group']
)
print("Table 1 — Baseline characteristics by 30-day readmission status:")
display(t1)


## 6. Statistical Tests for Group Differences

In [ ]:
print("Statistical tests: readmitted vs not readmitted\n")

# Mann-Whitney U — skewed continuous
for col, label in [('los_days','LOS'),('age','Age')]:
    g0 = df.loc[df['readmitted_30']==0, col].dropna()
    g1 = df.loc[df['readmitted_30']==1, col].dropna()
    sk = skew(df[col].dropna())
    if abs(sk) > 0.5:
        stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
        print(f"{label} (skewed → Mann-Whitney U): U={stat:.0f}, p={p:.4f}")
    else:
        t, p = ttest_ind(g0, g1)
        print(f"{label} (normal → t-test): t={t:.3f}, p={p:.4f}")

# Chi-square — categorical
for col in ['sex','payer','admit_type']:
    ct = pd.crosstab(df[col], df['readmitted_30'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = '*' if p < 0.05 else ''
    print(f"{col:12s} (chi-square): χ²={chi2:.2f}, df={dof}, p={p:.4f} {sig}")


## 7. Knowledge Check
1. Why is the Wilson interval preferred over the normal-approximation CI for proportions near 0 or 1?
2. After age-standardisation, Private insurance shows a *higher* readmission rate than crude. What does this mean?
3. When should you report median [IQR] instead of mean ± SD in a health outcomes table?
4. What does a significant chi-square for `payer × readmission` imply for the analysis plan?
5. Re-create Table 1 stratified by `sex` and interpret any striking differences.


In [ ]:
# Exercise 5 workspace
t1_sex = table1(df,'sex',['age','los_days','total_charge'],['payer','admit_type','age_group','readmitted_30'])
display(t1_sex)


---
**Next → NB-03: Visualising Clinical Distributions**